# NB05 — Multi-Ticker Ablation with Val-Tuned RF and Reduced Hierarchy Depth

Fork of `04_real_data_price_prediction_yfinance.ipynb` that addresses three weaknesses identified during NB04 analysis:

1. **Reduced hierarchy depth** — `max_depth=2` (4 leaves, ~800 train samples per leaf) instead of `max_depth=4` (16 leaves, ~200 per leaf). NB04 permutation importance suggested the 16-leaf decomposition overfit.
2. **Multiple tickers** — the comparison runs on `SPY`, `NVDA`, `BTC-USD` so the +0.9 pp indices win seen on SPY in NB04 can be checked for robustness.
3. **Validation set actually used** — RF `(max_depth, min_samples_leaf)` are picked per (ticker, feature-set, target) by score on the validation slice; the model is then refit on `train+val` before being scored on `test`. NB04 fit on `train` and went straight to `test`, leaving `validation_df` dead.

Three feature sets are compared per ticker: **baseline** (price/volume), **indices** (baseline + greed/fear scalars), **hierarchy** (indices + 4 soft-membership columns).

Each run appends a structured entry to `EXPERIMENTS.md` at the repo root.


In [1]:
import math
import os
import json
import random
from dataclasses import dataclass
from datetime import datetime
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    accuracy_score,
    balanced_accuracy_score,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

TICKERS = ["SPY", "NVDA", "BTC-USD"]
START_DATE = "2010-01-01"
END_DATE = None
HORIZON = 5
TEST_FRACTION = 0.20
VALIDATION_FRACTION = 0.15

MAX_HIERARCHY_DEPTH = 2
MIN_LEAF_COUNT = 200

RF_N_ESTIMATORS = 300
RF_GRID = [
    {"max_depth": md_, "min_samples_leaf": msl}
    for md_ in (4, 6, 8)
    for msl in (10, 20, 40)
]

JOURNAL_PATH = "EXPERIMENTS.md"


## Data loading and feature construction

Identical to NB04 — copied here so NB05 is self-contained.


In [2]:
def load_yfinance_ohlcv(ticker: str, start_date: str, end_date: Optional[str] = None) -> pd.DataFrame:
    raw = yf.download(
        tickers=ticker,
        start=start_date,
        end=end_date,
        auto_adjust=True,
        progress=False,
        group_by="column",
    )
    if raw.empty:
        raise RuntimeError(f"No data returned for ticker: {ticker}")
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = [col[0].lower() for col in raw.columns]
    else:
        raw.columns = [str(col).lower() for col in raw.columns]
    required = ["open", "high", "low", "close", "volume"]
    missing = [col for col in required if col not in raw.columns]
    if missing:
        raise RuntimeError(f"Missing required columns: {missing}")
    df = raw[required].copy().reset_index()
    df = df.rename(columns={"Date": "date", "Datetime": "date"})
    df["date"] = pd.to_datetime(df["date"])
    return df.dropna().reset_index(drop=True)


def rolling_zscore(x: pd.Series, window: int, eps: float = 1e-8) -> pd.Series:
    mean = x.rolling(window, min_periods=max(3, window // 3)).mean()
    std = x.rolling(window, min_periods=max(3, window // 3)).std()
    return (x - mean) / (std + eps)


def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-x))


def make_real_market_features(df: pd.DataFrame, horizon: int) -> pd.DataFrame:
    out = df.copy()
    for lag in (1, 2, 5, 10, 20):
        out[f"ret_{lag}"] = out["close"].pct_change(lag)
    out["range_pct"] = (out["high"] - out["low"]) / out["close"]
    out["open_close_pct"] = (out["close"] - out["open"]) / out["open"]
    for w in (5, 10, 20):
        out[f"vol_{w}"] = out["ret_1"].rolling(w).std()
    for w in (10, 20, 50):
        out[f"ma_{w}"] = out["close"].rolling(w).mean()
        out[f"price_to_ma_{w}"] = out["close"] / out[f"ma_{w}"] - 1.0
    out["volume_log"] = np.log1p(out["volume"])
    out["volume_z_20"] = rolling_zscore(out["volume_log"], 20)
    for w in (20, 50):
        out[f"rolling_max_{w}"] = out["close"].rolling(w).max()
        out[f"drawdown_{w}"] = out["close"] / out[f"rolling_max_{w}"] - 1.0
    out["trend_pressure"] = rolling_zscore(out["ret_10"], 30)
    out["fear_pressure"] = (
        -4.0 * out["drawdown_20"] - 2.0 * out["drawdown_50"]
        + 8.0 * out["vol_10"] + 0.35 * out["volume_z_20"]
        - 0.50 * out["ret_5"]
    )
    out["greed_pressure"] = (
        1.50 * out["trend_pressure"] + 2.00 * out["price_to_ma_20"]
        + 1.00 * out["ret_20"] + 0.20 * out["volume_z_20"]
        - 3.00 * out["vol_20"]
    )
    out["fear_index"] = sigmoid_np(out["fear_pressure"].clip(-8, 8))
    out["greed_index"] = sigmoid_np(out["greed_pressure"].clip(-8, 8))
    out["future_close"] = out["close"].shift(-horizon)
    out["future_return"] = out["future_close"] / out["close"] - 1.0
    out["future_direction"] = (out["future_return"] > 0).astype(int)
    out = out.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    return out


BASELINE_FEATURES = [
    "ret_1", "ret_2", "ret_5", "ret_10", "ret_20",
    "range_pct", "open_close_pct",
    "vol_5", "vol_10", "vol_20",
    "price_to_ma_10", "price_to_ma_20", "price_to_ma_50",
    "volume_z_20", "drawdown_20", "drawdown_50",
]


## Hierarchy (max_depth=2)

Same construction logic as NB04 but with `MAX_HIERARCHY_DEPTH=2`, yielding 4 leaves and ~800 training samples per leaf on the typical 16-year daily series.


In [3]:
@dataclass
class MeasureNode:
    node_id: str
    depth: int
    bounds: Dict[str, Tuple[float, float]]
    mass: float
    count: int
    mean: Dict[str, float]
    children: Optional[List["MeasureNode"]] = None

    @property
    def is_leaf(self) -> bool:
        return not self.children


def in_bounds(row: pd.Series, bounds: Dict[str, Tuple[float, float]]) -> bool:
    for col, (lo, hi) in bounds.items():
        v = float(row[col])
        if not (lo <= v < hi):
            return False
    return True


def split_node(node, split_col, split_value):
    lo, hi = node.bounds[split_col]
    lb, rb = dict(node.bounds), dict(node.bounds)
    lb[split_col] = (lo, split_value)
    rb[split_col] = (split_value, hi)
    node.children = [
        MeasureNode(node.node_id + "L", node.depth + 1, lb, node.mass / 2.0, 0, {}, None),
        MeasureNode(node.node_id + "R", node.depth + 1, rb, node.mass / 2.0, 0, {}, None),
    ]
    return node.children


def collect_leaves(node):
    if node.is_leaf:
        return [node]
    leaves = []
    for c in node.children:
        leaves.extend(collect_leaves(c))
    return leaves


def build_hierarchy(train_df, max_depth=2, min_count=200):
    feature_cols = ["greed_index", "fear_index"]
    root = MeasureNode(
        "root", 0,
        {"greed_index": (0.0, 1.000001), "fear_index": (0.0, 1.000001)},
        1.0, len(train_df),
        {c: float(train_df[c].mean()) for c in feature_cols}, None,
    )
    queue = [root]
    while queue:
        node = queue.pop(0)
        if node.depth >= max_depth:
            continue
        mask = train_df.apply(lambda r: in_bounds(r, node.bounds), axis=1)
        subset = train_df[mask]
        node.count = len(subset)
        node.mass = node.count / max(len(train_df), 1)
        if node.count < min_count:
            continue
        variances = {c: float(subset[c].var()) for c in feature_cols}
        split_col = max(variances, key=variances.get)
        split_value = float(subset[split_col].median())
        if node.bounds[split_col][0] < split_value < node.bounds[split_col][1]:
            queue.extend(split_node(node, split_col, split_value))
    for leaf in collect_leaves(root):
        mask = train_df.apply(lambda r: in_bounds(r, leaf.bounds), axis=1)
        subset = train_df[mask]
        leaf.count = len(subset)
        leaf.mass = leaf.count / max(len(train_df), 1)
        leaf.mean = {c: (float(subset[c].mean()) if len(subset) else 0.0) for c in feature_cols}
    return root


def soft_membership(row, leaves, temperature=0.04):
    point = np.array([float(row["greed_index"]), float(row["fear_index"])])
    centers = np.array([[(l.bounds["greed_index"][0] + l.bounds["greed_index"][1]) / 2.0,
                          (l.bounds["fear_index"][0] + l.bounds["fear_index"][1]) / 2.0]
                         for l in leaves])
    distances = np.sum((centers - point[None, :]) ** 2, axis=1)
    logits = -distances / max(temperature, 1e-8)
    logits = logits - logits.max()
    w = np.exp(logits)
    return w / w.sum()


def add_memberships(frame, leaves, membership_columns):
    matrix = np.vstack([soft_membership(row, leaves) for _, row in frame.iterrows()])
    df_m = pd.DataFrame(matrix, columns=membership_columns, index=frame.index)
    return pd.concat([frame.reset_index(drop=True), df_m.reset_index(drop=True)], axis=1)


def time_split(frame, validation_fraction, test_fraction):
    n = len(frame)
    test_start = int(n * (1.0 - test_fraction))
    validation_start = int(test_start * (1.0 - validation_fraction))
    return (
        frame.iloc[:validation_start].copy(),
        frame.iloc[validation_start:test_start].copy(),
        frame.iloc[test_start:].copy(),
    )


## Validation-tuned fit + evaluate

For each feature set we grid-search RF `(max_depth, min_samples_leaf)` on the validation slice, then refit on `train+val` with the chosen hyperparameters before scoring on `test`. Scoring metric: balanced accuracy for the classifier, negative MAE for the regressor.

This is a small grid (9 combos × 2 models = 18 RF fits per feature set, 54 per ticker, ~160 over 3 tickers). With `n_jobs=-1` and `n_estimators=300` this runs in a couple of minutes on a multi-core CPU.


In [4]:
def _rf_reg(params):
    return RandomForestRegressor(
        n_estimators=RF_N_ESTIMATORS,
        random_state=SEED, n_jobs=-1,
        **params,
    )

def _rf_cls(params):
    return RandomForestClassifier(
        n_estimators=RF_N_ESTIMATORS,
        random_state=SEED, n_jobs=-1,
        class_weight="balanced_subsample",
        **params,
    )


def select_rf_params(builder, x_train, y_train, x_val, y_val, scoring):
    best_params, best_score = None, -np.inf
    for params in RF_GRID:
        m = builder(params)
        m.fit(x_train, y_train)
        s = scoring(m, x_val, y_val)
        if s > best_score:
            best_params, best_score = params, s
    return best_params, best_score


def fit_with_val_tuning(feature_columns, train_df, val_df, test_df, model_name):
    x_train = train_df[feature_columns]
    x_val = val_df[feature_columns]
    x_test = test_df[feature_columns]

    y_train_r = train_df["future_return"]; y_val_r = val_df["future_return"]; y_test_r = test_df["future_return"]
    y_train_c = train_df["future_direction"]; y_val_c = val_df["future_direction"]; y_test_c = test_df["future_direction"]

    reg_score = lambda m, xv, yv: -mean_absolute_error(yv, m.predict(xv))
    cls_score = lambda m, xv, yv: balanced_accuracy_score(yv, m.predict(xv))

    best_reg_params, val_neg_mae = select_rf_params(_rf_reg, x_train, y_train_r, x_val, y_val_r, reg_score)
    best_cls_params, val_bal_acc = select_rf_params(_rf_cls, x_train, y_train_c, x_val, y_val_c, cls_score)

    x_trainval = pd.concat([x_train, x_val], axis=0)
    y_trainval_r = pd.concat([y_train_r, y_val_r], axis=0)
    y_trainval_c = pd.concat([y_train_c, y_val_c], axis=0)

    reg = _rf_reg(best_reg_params); reg.fit(x_trainval, y_trainval_r)
    cls = _rf_cls(best_cls_params); cls.fit(x_trainval, y_trainval_c)

    pred_return = reg.predict(x_test)
    pred_dir = cls.predict(x_test)
    pred_dir_from_reg = (pred_return > 0).astype(int)

    result = {
        "model": model_name,
        "rf_reg_params": best_reg_params,
        "rf_cls_params": best_cls_params,
        "val_neg_mae": float(val_neg_mae),
        "val_bal_acc": float(val_bal_acc),
        "test_return_mae": float(mean_absolute_error(y_test_r, pred_return)),
        "test_return_rmse": float(math.sqrt(mean_squared_error(y_test_r, pred_return))),
        "test_dir_acc": float(accuracy_score(y_test_c, pred_dir)),
        "test_bal_acc": float(balanced_accuracy_score(y_test_c, pred_dir)),
        "test_dir_acc_from_reg": float(accuracy_score(y_test_c, pred_dir_from_reg)),
        "test_naive_mae": float(mean_absolute_error(y_test_r, np.zeros_like(y_test_r))),
        "test_majority_acc": float(max(y_test_c.mean(), 1.0 - y_test_c.mean())),
    }
    return result, (reg, cls)


## Per-ticker pipeline

Loads OHLCV, builds features, fits the hierarchy on training data only, then runs the three-way ablation with validation tuning.


In [5]:
def run_ticker(ticker, start_date=START_DATE, end_date=END_DATE):
    df = load_yfinance_ohlcv(ticker, start_date, end_date)
    feat = make_real_market_features(df, HORIZON)
    train_df, val_df, test_df = time_split(feat, VALIDATION_FRACTION, TEST_FRACTION)

    root = build_hierarchy(train_df, max_depth=MAX_HIERARCHY_DEPTH, min_count=MIN_LEAF_COUNT)
    leaves = collect_leaves(root)
    membership_columns = [f"leaf_{leaf.node_id}" for leaf in leaves]

    train_m = add_memberships(train_df, leaves, membership_columns)
    val_m = add_memberships(val_df, leaves, membership_columns)
    test_m = add_memberships(test_df, leaves, membership_columns)

    indices_features = BASELINE_FEATURES + ["greed_index", "fear_index"]
    hierarchy_features = indices_features + membership_columns

    feature_sets = [
        (BASELINE_FEATURES, "baseline"),
        (indices_features, "indices"),
        (hierarchy_features, "hierarchy"),
    ]

    rows = []
    fitted = {}
    for fs, name in feature_sets:
        res, models = fit_with_val_tuning(fs, train_m, val_m, test_m, name)
        res["ticker"] = ticker
        res["n_train"] = len(train_m)
        res["n_val"] = len(val_m)
        res["n_test"] = len(test_m)
        res["n_leaves"] = len(leaves)
        rows.append(res)
        fitted[name] = (models, fs)

    return pd.DataFrame(rows), fitted, test_m, membership_columns


all_results = []
all_fitted = {}
all_test = {}
all_membership_columns = {}
for tk in TICKERS:
    print(f"Running {tk}...")
    res_df, fitted, test_m, mcols = run_ticker(tk)
    all_results.append(res_df)
    all_fitted[tk] = fitted
    all_test[tk] = test_m
    all_membership_columns[tk] = mcols

agg = pd.concat(all_results, ignore_index=True)
agg[["ticker", "model", "n_train", "n_val", "n_test", "n_leaves",
     "test_return_mae", "test_dir_acc", "test_bal_acc", "test_dir_acc_from_reg",
     "test_majority_acc", "test_naive_mae"]]


Running SPY...


Running NVDA...


Running BTC-USD...


,ticker,model,n_train,n_val,n_test,n_leaves,test_return_mae,test_dir_acc,test_bal_acc,test_dir_acc_from_reg,test_majority_acc,test_naive_mae
0,SPY,baseline,2759,488,812,4,0.014597,0.564039,0.513842,0.623153,0.628079,0.015256
1,SPY,indices,2759,488,812,4,0.014603,0.541872,0.500922,0.624384,0.628079,0.015256
2,SPY,hierarchy,2759,488,812,4,0.014603,0.545567,0.502513,0.623153,0.628079,0.015256
3,NVDA,baseline,2759,488,812,4,0.050727,0.548030,0.503450,0.572660,0.599754,0.050854
4,NVDA,indices,2759,488,812,4,0.050669,0.543103,0.497808,0.576355,0.599754,0.050854
5,NVDA,hierarchy,2759,488,812,4,0.050638,0.539409,0.496263,0.578818,0.599754,0.050854
6,BTC-USD,baseline,2856,505,841,4,0.042339,0.510107,0.525847,0.526754,0.544590,0.040607
7,BTC-USD,indices,2856,505,841,4,0.042392,0.500595,0.516259,0.537455,0.544590,0.040607
8,BTC-USD,hierarchy,2856,505,841,4,0.042410,0.502973,0.517801,0.523187,0.544590,0.040607


## Cross-ticker summary

Pivot the key test-set metrics by `ticker × model`.


In [6]:
metrics = ["test_return_mae", "test_dir_acc", "test_bal_acc", "test_dir_acc_from_reg"]
summary = agg.pivot_table(index="model", columns="ticker", values=metrics)
summary = summary.reorder_levels([1, 0], axis=1).sort_index(axis=1)
summary


ticker         BTC-USD                                                     \
          test_bal_acc test_dir_acc test_dir_acc_from_reg test_return_mae   
model                                                                       
baseline      0.525847     0.510107              0.526754        0.042339   
hierarchy     0.517801     0.502973              0.523187        0.042410   
indices       0.516259     0.500595              0.537455        0.042392   

ticker            NVDA                                                     \
          test_bal_acc test_dir_acc test_dir_acc_from_reg test_return_mae   
model                                                                       
baseline      0.503450     0.548030              0.572660        0.050727   
hierarchy     0.496263     0.539409              0.578818        0.050638   
indices       0.497808     0.543103              0.576355        0.050669   

ticker             SPY                                                     
          test_bal_acc test_dir_acc test_dir_acc_from_reg test_return_mae  
model                                                                      
baseline      0.513842     0.564039              0.623153        0.014597  
hierarchy     0.502513     0.545567              0.623153        0.014603  
indices       0.500922     0.541872              0.624384        0.014603

In [7]:
delta = agg.set_index(["ticker", "model"])[["test_return_mae", "test_bal_acc", "test_dir_acc", "test_dir_acc_from_reg"]]
baseline_rows = delta.xs("baseline", level="model")
delta_indices = delta.xs("indices", level="model") - baseline_rows
delta_hierarchy = delta.xs("hierarchy", level="model") - baseline_rows
deltas = pd.concat({"indices_minus_baseline": delta_indices, "hierarchy_minus_baseline": delta_hierarchy}, axis=1)
deltas


indices_minus_baseline                            \
               test_return_mae test_bal_acc test_dir_acc   
ticker                                                     
SPY                   0.000006    -0.012920    -0.022167   
NVDA                 -0.000058    -0.005642    -0.004926   
BTC-USD               0.000053    -0.009589    -0.009512   

                              hierarchy_minus_baseline               \
        test_dir_acc_from_reg          test_return_mae test_bal_acc   
ticker                                                                
SPY                  0.001232                 0.000007    -0.011330   
NVDA                 0.003695                -0.000090    -0.007187   
BTC-USD              0.010702                 0.000071    -0.008047   

                                            
        test_dir_acc test_dir_acc_from_reg  
ticker                                      
SPY        -0.018473              0.000000  
NVDA       -0.008621              0.006158  
BTC-USD    -0.007134             -0.003567

## Permutation importance on the indices model

Per-ticker permutation importance for the val-tuned `indices` classifier (the one that won on SPY in NB04). Positive numbers mean the feature carries genuine held-out signal for balanced accuracy.


In [8]:
def perm_table(model, x_test, y_test, feature_columns, scoring, n_repeats=10):
    pi = permutation_importance(
        model, x_test, y_test,
        n_repeats=n_repeats, random_state=SEED, n_jobs=-1,
        scoring=scoring,
    )
    return pd.DataFrame({
        "feature": feature_columns,
        "perm_importance_mean": pi.importances_mean,
        "perm_importance_std": pi.importances_std,
    }).sort_values("perm_importance_mean", ascending=False).reset_index(drop=True)


perm_importance_by_ticker = {}
for tk in TICKERS:
    (reg, cls), fs = all_fitted[tk]["indices"]
    test_m = all_test[tk]
    perm = perm_table(cls, test_m[fs], test_m["future_direction"], fs, scoring="balanced_accuracy")
    perm_importance_by_ticker[tk] = perm
    print(f"=== {tk} ===")
    print(perm.to_string(index=False))
    print()


=== SPY ===
       feature  perm_importance_mean  perm_importance_std
        ret_20              0.007312             0.006650
price_to_ma_50              0.003630             0.009417
         ret_2              0.003261             0.008679
   drawdown_20              0.002770             0.006665
open_close_pct              0.002667             0.007425
price_to_ma_20              0.002091             0.004409
   volume_z_20              0.001400             0.004892
        ret_10             -0.000039             0.005634
    fear_index             -0.000401             0.005264
         ret_5             -0.001956             0.005257
   drawdown_50             -0.004210             0.003940
price_to_ma_10             -0.004742             0.004956
         ret_1             -0.005806             0.002650
         vol_5             -0.005841             0.006973
     range_pct             -0.005931             0.008466
        vol_20             -0.008056             0.007147
  

=== NVDA ===
       feature  perm_importance_mean  perm_importance_std
        ret_20              0.019666             0.006563
price_to_ma_50              0.007646             0.004184
         ret_2              0.004568             0.003240
   drawdown_50              0.002213             0.004786
         ret_1              0.001532             0.003170
         ret_5              0.000980             0.005061
        vol_10              0.000599             0.010441
   greed_index             -0.000087             0.006661
price_to_ma_20             -0.000463             0.004847
price_to_ma_10             -0.000923             0.003596
        ret_10             -0.001486             0.004987
    fear_index             -0.001947             0.001790
open_close_pct             -0.002465             0.003463
   drawdown_20             -0.003432             0.004455
   volume_z_20             -0.004308             0.004257
     range_pct             -0.008569             0.004262
 

=== BTC-USD ===
       feature  perm_importance_mean  perm_importance_std
price_to_ma_50              0.015995             0.008539
         ret_5              0.011886             0.003723
   drawdown_20              0.009527             0.002419
        vol_10              0.005562             0.004630
         vol_5              0.003833             0.004016
        ret_20              0.003568             0.005115
   greed_index              0.001686             0.006211
   volume_z_20              0.001531             0.001537
    fear_index              0.000482             0.002370
        ret_10             -0.000225             0.004369
         ret_2             -0.001142             0.001310
   drawdown_50             -0.001148             0.006369
price_to_ma_10             -0.001400             0.005354
     range_pct             -0.002720             0.002951
         ret_1             -0.004068             0.002091
price_to_ma_20             -0.004784             0.00817

## Append run to EXPERIMENTS.md

Structured journal entry with config, summary table, and per-ticker delta vs baseline.


In [9]:
def append_to_journal(agg_df, deltas_df, journal_path=JOURNAL_PATH):
    header_needed = not os.path.exists(journal_path)
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    config = {
        "notebook": "05_multi_ticker_validation_tuned.ipynb",
        "tickers": TICKERS,
        "start_date": START_DATE,
        "horizon": HORIZON,
        "test_fraction": TEST_FRACTION,
        "validation_fraction": VALIDATION_FRACTION,
        "max_hierarchy_depth": MAX_HIERARCHY_DEPTH,
        "min_leaf_count": MIN_LEAF_COUNT,
        "rf_n_estimators": RF_N_ESTIMATORS,
        "rf_grid": RF_GRID,
        "seed": SEED,
    }

    cols_keep = ["ticker", "model", "n_train", "n_val", "n_test", "n_leaves",
                 "test_return_mae", "test_dir_acc", "test_bal_acc",
                 "test_dir_acc_from_reg", "test_majority_acc",
                 "rf_reg_params", "rf_cls_params"]
    table = agg_df[cols_keep].copy()
    for c in ("test_return_mae",):
        table[c] = table[c].map(lambda v: f"{v:.5f}")
    for c in ("test_dir_acc", "test_bal_acc", "test_dir_acc_from_reg", "test_majority_acc"):
        table[c] = table[c].map(lambda v: f"{v:.4f}")

    parts = []
    if header_needed:
        parts.append("# Experiments journal\n\n")
        parts.append("Append-only log of model-comparison runs. Newest entries at the bottom.\n")
    parts.append(f"\n---\n")
    parts.append(f"\n## {timestamp} — NB05 multi-ticker ablation (val-tuned RF, hierarchy max_depth={MAX_HIERARCHY_DEPTH})\n")
    parts.append(f"\n**Config**\n\n```json\n{json.dumps(config, indent=2)}\n```\n")
    parts.append(f"\n**Per-ticker test-set results**\n\n")
    parts.append(table.to_markdown(index=False))
    parts.append(f"\n\n**Deltas vs baseline (per ticker)**\n\n")
    parts.append(deltas_df.round(5).to_markdown())
    parts.append("\n")
    text = "".join(parts)

    with open(journal_path, "a") as f:
        f.write(text)

    print(f"Appended entry to {journal_path}")
    return text


journal_text = append_to_journal(agg, deltas)
print()
print(journal_text)


Appended entry to EXPERIMENTS.md

# Experiments journal

Append-only log of model-comparison runs. Newest entries at the bottom.

---

## 2026-05-12 15:12:40 — NB05 multi-ticker ablation (val-tuned RF, hierarchy max_depth=2)

**Config**

```json
{
  "notebook": "05_multi_ticker_validation_tuned.ipynb",
  "tickers": [
    "SPY",
    "NVDA",
    "BTC-USD"
  ],
  "start_date": "2010-01-01",
  "horizon": 5,
  "test_fraction": 0.2,
  "validation_fraction": 0.15,
  "max_hierarchy_depth": 2,
  "min_leaf_count": 200,
  "rf_n_estimators": 300,
  "rf_grid": [
    {
      "max_depth": 4,
      "min_samples_leaf": 10
    },
    {
      "max_depth": 4,
      "min_samples_leaf": 20
    },
    {
      "max_depth": 4,
      "min_samples_leaf": 40
    },
    {
      "max_depth": 6,
      "min_samples_leaf": 10
    },
    {
      "max_depth": 6,
      "min_samples_leaf": 20
    },
    {
      "max_depth": 6,
      "min_samples_leaf": 40
    },
    {
      "max_depth": 8,
      "min_samples_leaf": 10
   

## What to look for

1. **Does `indices` beat `baseline` on test balanced accuracy across all three tickers?** If yes, the +0.9 pp finding on SPY in NB04 was not a one-ticker artifact.
2. **Does `hierarchy` (now with only 4 leaves) close the gap to `indices`?** If yes, depth was the culprit in NB04. If the gap remains, the soft-membership decomposition adds nothing even at low cardinality.
3. **Does the regression MAE gap shrink?** NB04 had hierarchy ≈ indices ≈ baseline on regression — with val tuning + fewer leaves we should see if any single feature set pulls ahead.
4. **Permutation importance on the indices classifier:** if `greed_index` and `fear_index` are consistently in the top half across tickers, the indices contribution is robust. If they're near zero / negative on BTC-USD or NVDA, the signal is regime-specific.
